In [30]:

import copy
import struct
from enum import IntEnum
from typing import Optional, Tuple, List
import crcmod  # pip install crcmod
import unittest
from frame import Frame, FrameType
from hamming import Hamming74
from encryptedFrame import EncryptedFrame


### проверка без помех

In [4]:
data_to_send = b"Secret Message"

frame_src = EncryptedFrame(
    dest_addr=0x02, 
    src_addr=0x01, 
    frame_type=FrameType.DATA, 
    data=data_to_send
)


In [ ]:
encrypted = frame_src.encrypted_serialize()
encrypted

bytes

In [12]:
frame_dest = EncryptedFrame(raw_bytes=encrypted)
frame_dest.data

b'Secret Message'

### проверка с помехами

In [25]:
data_to_send = "Всем привет".encode("utf-8")
print(type(data_to_send))
data_to_send

<class 'bytes'>


b'\xd0\x92\xd1\x81\xd0\xb5\xd0\xbc \xd0\xbf\xd1\x80\xd0\xb8\xd0\xb2\xd0\xb5\xd1\x82'

In [23]:
frame_src = EncryptedFrame(
    dest_addr=0x03, 
    src_addr=0x01, 
    frame_type=FrameType.DATA, 
    data=data_to_send
)

wire_bytes = frame_src.encrypted_serialize()


In [55]:
changed_bytes = bytearray(wire_bytes)
changed_bytes[10] ^= 0x04
changed_bytes[31] ^= 0x04
print(wire_bytes)
print(bytes(changed_bytes))

b'~\x00p\x00<\x00<\x1e\xa8\xd8\x00\x92L\xd8<\x8c<\xd8\x00\xb4\xa8\xd8\x00\xb5\x8c&\x00\xd8\x00\xb5\xfc\xd8<\x8c\x00\xd8\x00\xb5\x18\xd8\x00\xb4L\xd8\x00\xb4\xa8\xd8<\x8cL\xc7\xb0\xc6\xa8~'
b'~\x00p\x00<\x00<\x1e\xa8\xd8\x04\x92L\xd8<\x8c<\xd8\x00\xb4\xa8\xd8\x00\xb5\x8c&\x00\xd8\x00\xb5\xfc\xdc<\x8c\x00\xd8\x00\xb5\x18\xd8\x00\xb4L\xd8\x00\xb4\xa8\xd8<\x8cL\xc7\xb0\xc6\xa8~'


In [56]:
EncryptedFrame(raw_bytes=wire_bytes).data.decode("utf-8")

'Всем привет'